# 4. Feature selection with deviance-based and scran-normalized metrics

After batch correction, the next step is to identify informative genes for downstream dimensionality reduction and clustering. Using all genes is unnecessary and often harmful, as many genes are either lowly expressed, noisy, or uninformative for cell identity.

In this notebook, we:

- perform feature selection using the **scry** package in R via deviance-based ranking (`devianceFeatureSelection`),
- select the top 4,000 genes by binomial deviance,
- annotate these genes within the AnnData object, and
- compute standard highly variable genes (HVGs) using Scanpy on the scran-normalized layer.

The output is an AnnData object with feature-selection annotations that can be used as input for subsequent PCA, UMAP, clustering, and reclustering analyses.

## 4.1. Imports, reproducibility settings, and file paths

In [ ]:
import copy
import logging
import warnings
import anndata2ri
import matplotlib.pyplot as plt
import numpy as np
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc
import seaborn as sns
import scipy.sparse
from memory_profiler import profile  # optional, for debugging memory usage
from rpy2.robjects.conversion import localconverter

# Suppress warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Reproducibility and logging
SEED = 123
np.random.seed(SEED)
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
rcb.logger.setLevel(logging.ERROR)

# File paths
input_file = "combined_8_samples_batch_corrected.h5ad"
output_file = "feature_selected.h5ad"

print(f"Input file:  {input_file}")
print(f"Output file: {output_file}")

## 4.2. Load ComBat-corrected dataset

We load the ComBat-corrected AnnData object containing all 8 samples, which now serves as the basis for gene-level feature selection.

In [ ]:
adata = sc.read_h5ad(input_file)
adata

## 4.3. Deviance-based feature selection with scry (R)

We use the R package **scry** to perform deviance-based feature selection. The function `devianceFeatureSelection` computes a binomial deviance statistic per gene, which captures how much a gene deviates from a null model of expression. Genes with higher deviance are considered more informative for variation across cells.

Here, we:

1. activate the AnnData ↔ R conversion bridge,
2. pass the AnnData object into R,
3. run `devianceFeatureSelection` using the main assay (`X`), and
4. retrieve the per-gene binomial deviance values back into Python.

In [ ]:
# Activate AnnData ↔ R conversion
anndata2ri.activate()

with localconverter(anndata2ri.converter):
    ro.globalenv["adata"] = adata
    ro.r("""
        library(scry)
        sce <- devianceFeatureSelection(adata, assay = "X")
        binomial_deviance <- rowData(sce)$binomial_deviance
    """)

binomial_deviance = np.array(ro.r("binomial_deviance"))
print(f"Retrieved binomial deviance for {binomial_deviance.shape[0]} genes.")

## 4.4. Select top deviant genes and annotate AnnData

We rank genes by their binomial deviance and select the top 4,000 as our deviance-based feature set. We then annotate:

- `var["binomial_deviance"]`: numeric deviance score per gene
- `var["highly_deviant"]`: boolean flag indicating membership in the top 4,000 deviant genes

In [ ]:
# Select top N genes by deviance
n_top = 4000
idx = binomial_deviance.argsort()[-n_top:]

mask = np.zeros(adata.var_names.shape, dtype=bool)
mask[idx] = True

# Annotate AnnData
adata.var["binomial_deviance"] = binomial_deviance
adata.var["highly_deviant"] = mask

print(f"Marked {mask.sum()} genes as highly deviant (top {n_top}).")

## 4.5. Highly variable genes using scran-normalized counts

In addition to deviance-based selection, we also compute highly variable genes using Scanpy’s `pp.highly_variable_genes`, applied to the `scran_normalization` layer. This gives a second, more conventional feature set based on mean–variance relationships under the scran normalization.

We keep both annotations so that downstream notebooks can choose between deviance-based and HVG-based feature sets or use their intersection.

In [ ]:
sc.pp.highly_variable_genes(adata, layer="scran_normalization")
print(f"HVGs (Scanpy, scran_normalization): {adata.var['highly_variable'].sum()} genes flagged.")

## 4.6. Visualize feature selection

We visualize the mean–dispersion relationship and highlight the genes selected as highly deviant. This provides a quick sanity check that the selected genes span a range of expression levels and dispersions rather than being dominated by low-coverage noise.

In [ ]:
plt.figure(figsize=(8, 6))
ax = sns.scatterplot(
    data=adata.var,
    x="means",
    y="dispersions",
    hue="highly_deviant",
    s=5,
    palette={True: "red", False: "grey"},
    legend=False,
)
plt.title("Feature selection (red: highly deviant genes)")
plt.xlabel("Mean expression")
plt.ylabel("Dispersion")
plt.tight_layout()
plt.show()

## 4.7. Save annotated AnnData and next steps

We now save the ComBat-corrected AnnData object with additional annotations:

- `var["binomial_deviance"]`: deviance-based feature statistic from scry,
- `var["highly_deviant"]`: deviance-based feature selection flag,
- `var["highly_variable"]`: Scanpy HVG flag (based on `scran_normalization`).

This object will be used in the next notebook for dimensionality reduction (PCA/UMAP) and clustering, where we can choose which feature set (or combination) to use.

In [ ]:
adata.write(output_file)

print("Feature selection complete.")
print(f"Saved annotated AnnData to: {output_file}")